# Faruq-v3 AF2 + FFAB2 Selectivity Follow-up — Kaggle

Follow-up setelah matched from-start FFAB2 REJECT. Notebook ini menjalankan: per-class delta, beta sweep, P3/P4/P5 bypass, parent-residual interpolation, ambiguity gating, lalu **hanya jika diagnostic gate PASS** melakukan fresh 3-seed selective retraining dan frozen confirmation. Test tetap terkunci.

Required inputs: (1) `faruq-v3-experiment-core-v1` terbaru; (2) output/state dari notebook `Faruq_V3_AF2_FFAB2_All_Seeds_All_Stages_Kaggle` yang memuat 3 AF2FS + 3 AF2FFAB2FS results/checkpoints. GPU + Internet ON.

In [ ]:
from pathlib import Path
import importlib,json,os,shutil,subprocess,sys,time,torch,zipfile
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir(): raise RuntimeError('Kaggle-only notebook')
for name in ('af2_spectral_kaggle_manifest.json','faruq-development-v3-grouped.tar.bin','D0_seed42_best.pt','D0_seed123_best.pt','D0_seed2026_best.pt'):
    m=sorted(INPUT.rglob(name))
    if len(m)!=1: raise FileNotFoundError(f'Harus ada tepat satu {name}; ditemukan {m}')
    print('CORE INPUT OK:',name,m[0])
STATE_NAME='af2-ffab2-all-seeds-all-stages-state.zip'
states=sorted(INPUT.rglob(STATE_NAME))
if len(states)>1: raise RuntimeError(f'State ambigu: {states}')
RESTORED=WORK/'af2-ffab2-prior-restored'
if states:
    if RESTORED.exists(): shutil.rmtree(RESTORED)
    RESTORED.mkdir(parents=True)
    with zipfile.ZipFile(states[0],'r') as z: z.extractall(RESTORED)
    print('RESTORED PRIOR STATE:',states[0])
else:
    RESTORED=INPUT
    print('No state ZIP; searching attached saved output directly')

In [ ]:
BRANCH='codex/af2-ffab2-selective-refinement'
REPO=WORK/'coffee-bean-detection'
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(1,4):
    r=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==3: raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for name in list(sys.modules):
    if name=='coffee_detector' or name.startswith('coffee_detector.'): sys.modules.pop(name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('BRANCH:',BRANCH)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan Kaggle GPU')
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_af2_ffa.py','tests/test_af2_ffa_from_start_dct.py','tests/test_af2_ffab2_selective_refinement.py'],cwd=REPO,check=True)

In [ ]:
from coffee_detector.experiments.prepare_af2_spectral_kaggle import prepare_af2_spectral_kaggle_input
DATA,ARTIFACTS,CORE=prepare_af2_spectral_kaggle_input(INPUT,WORK)
if CORE.get('decision')!='PASS' or CORE.get('test_images_accessed') is not False: raise RuntimeError('Core contract gagal')
if (DATA/'test').exists(): raise RuntimeError('TEST TEREXPOSE — STOP')
GROUPED=DATA/'faruq_grouped_summary.json'
SEEDS=(42,123,2026)
D0={s:Path(ARTIFACTS[f'D0_seed{s}_best.pt']) for s in SEEDS}
def find_one(filename):
    roots=(RESTORED,INPUT) if RESTORED!=INPUT else (INPUT,)
    matches=[]
    for root in roots: matches.extend(root.rglob(filename))
    unique=[]
    for p in matches:
        rp=p.resolve()
        if rp not in unique: unique.append(rp)
    if len(unique)!=1: raise FileNotFoundError(f'Harus tepat satu {filename}; ditemukan {unique}')
    return unique[0]
AF2_RESULTS=[find_one(f'AF2FS_seed{s}_result.json') for s in SEEDS]
FFAB_RESULTS=[find_one(f'AF2FFAB2FS_seed{s}_result.json') for s in SEEDS]
print('AF2 RESULTS:',AF2_RESULTS)
print('FFAB2 RESULTS:',FFAB_RESULTS)
OUT=WORK/'af2-ffab2-selective-refinement-v1'; OUT.mkdir(parents=True,exist_ok=True)
DIAG=OUT/'selectivity_analysis.json'

In [ ]:
# Stage A: no-training diagnosis. This may run many validation passes.
cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_ffab2_selectivity_analysis']
for p in AF2_RESULTS: cmd += ['--af2-result',str(p)]
for p in FFAB_RESULTS: cmd += ['--ffab2-result',str(p)]
cmd += ['--checkpoint-root',str(RESTORED),'--checkpoint-root',str(INPUT),'--data-root',str(DATA),'--output',str(DIAG),'--device','0']
subprocess.run(cmd,cwd=REPO,check=True)
diagnostic=json.loads(DIAG.read_text(encoding='utf-8'))
print('DIAGNOSTIC DECISION:',diagnostic['decision'])
print('TRAINING AUTHORIZED:',diagnostic['training_authorized'])
print('SELECTED:',json.dumps(diagnostic.get('selected_candidate'),indent=2))
print('CONSISTENT HELP CLASSES:',[(r['class'],round(100*r['mean_delta'],3)) for r in diagnostic['per_class']['consistent_help']])
print('CONSISTENT HARM CLASSES:',[(r['class'],round(100*r['mean_delta'],3)) for r in diagnostic['per_class']['consistent_harm']])

In [ ]:
# Stage B: only one selected candidate; only if the frozen diagnostic gate passed.
SELECTIVE_RESULTS=[]; FINAL_DECISION=None
if diagnostic.get('training_authorized') is True and diagnostic.get('decision')=='DIAGNOSTIC_CANDIDATE_FOUND':
    from coffee_detector.af2_ffa import run_af2_ffa_from_start_static_audit
    STATIC={}
    for s in SEEDS:
        p=OUT/'static_audits'/f'from_start_static_audit_seed{s}.json'; p.parent.mkdir(parents=True,exist_ok=True)
        audit=run_af2_ffa_from_start_static_audit(D0[s],p,device='cuda:0')
        if audit.get('decision')!='PASS': raise RuntimeError(f'static audit seed {s} gagal')
        STATIC[s]=p
    for s in SEEDS:
        cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_ffab2_selective_arm',
             '--data-root',str(DATA),'--grouped-summary',str(GROUPED),'--d0-checkpoint',str(D0[s]),
             '--static-audit',str(STATIC[s]),'--diagnostic',str(DIAG),'--output-root',str(OUT),
             '--seed',str(s),'--device','0','--authorize-training']
        subprocess.run(cmd,cwd=REPO,check=True)
        SELECTIVE_RESULTS.append(OUT/'val_reports'/f'AF2FFASR1_seed{s}_result.json')
    FINAL_DECISION=OUT/'val_reports'/'af2_ffab2_selective_decision.json'
    cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_ffab2_selective_decision',
         '--af2',*[str(p) for p in AF2_RESULTS],'--selective',*[str(p) for p in SELECTIVE_RESULTS],'--output',str(FINAL_DECISION)]
    subprocess.run(cmd,cwd=REPO,check=True)
    print(json.dumps(json.loads(FINAL_DECISION.read_text()),indent=2))
else:
    print('NO SELECTIVE TRAINING: diagnostic gate did not authorize it.')
archive=Path(shutil.make_archive(str(WORK/'af2-ffab2-selective-refinement-output'),'zip',root_dir=WORK,base_dir=OUT.name))
print('OUTPUT ZIP:',archive,archive.stat().st_size,'bytes')
print('TEST: LOCKED / NEVER OPENED')